In [ ]:
# Parameters (do not containerize this cell)
# =====
# Input
# -----
# user define/upload
# param_file_seagrass_site_data        = "[Seagrass_site_data.xlsx]"
param_file_seagrass_site_data        = "Seagrass_site_data.xlsx"
# exist/download
param_file_bottomT_p95               <- "bottomT_p95_daily_C.nc"
param_file_uo_mean_1.5m_m_s          <- "uo_mean_1.5m_m_s.nc"
param_file_vo_p90_1.5m_m_s           <- "vo_p90_1.5m_m_s.nc"
param_file_po4_mean_1.5m_mmol_m3     <- "po4_mean_monthly_1.5m_mmol_m3.nc"
param_file_pH_mean_1.5m              <- "pH_mean_monthly_1.5m.nc"
param_file_wave_height_VHM0_p95_m    <- "wave_height_p95_m.nc"
param_file_Surf_fgco2_p95_molC_m2_yr <- "Surf_fgco2_p95_molC_m2_yr.nc"
param_file_KD                        <- "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.KD.Kd_490.4km.nc"
param_file_RRS443                    <- "S3A_OLCI_ERRNT.20160425_20241231.L3m.CU.RRS.Rrs_443.4km.nc"

# Output
# -----
# temporary
param_file_SG_modeling_dataframe <- "SG_modeling_dataframe.csv"
# results
param_file_result <- "Seagrass_site_result.txt"

# User
# -----
param_user_email = "[the e-mail address you used to log into NaaVRE]"
# user define/upload, 1: dummy data; 0: to define/upload
param_use_dummy_data <- 1
# exist/download, 1: existing/downloaded data; 0: to download
param_use_exist_data <- 1

# Temporary
# -----
conf_directory_temporary_data <- "/tmp/data"

# MINIO
# -----
conf_minio_endpoint <- "scruffy.lab.uvalight.net:9000"
# conf_minio_endpoint <- "scruffy.lab.uvalight.net:9001"
conf_minio_region   <- "nl-uvalight"
conf_minio_bucket_public <- "naa-vre-public"
conf_minio_bucket_user   <- "naa-vre-user-data"
conf_minio_bucket_public_path <- "vl-bluecarbon"

In [ ]:
# Secrets (do not containerize this cell)
# =====
library("SecretsProvider")

secretsProvider <- SecretsProvider()

secret_minio_access_key = ""
secret_minio_access_key = secretsProvider$get_secret("secret_minio_access_key")

secret_minio_secret_key = ""
secret_minio_secret_key = secretsProvider$get_secret("secret_minio_secret_key")

In [ ]:
# MinIO data retriever
# =====
library("aws.s3")

Sys.setenv(
    "AWS_S3_ENDPOINT"    = conf_minio_endpoint,
    "AWS_DEFAULT_REGION" = conf_minio_region,
    "AWS_ACCESS_KEY_ID"     = secret_minio_access_key,
    "AWS_SECRET_ACCESS_KEY" = secret_minio_secret_key
)

In [ ]:
# Prepare
# =====
# Load dependencies
library(tidyverse)
library(RNetCDF)

# Ensure the temporary data storage directory exists
dir.create(conf_directory_temporary_data, showWarnings = FALSE)

In [ ]:
# Seagrass site data
# =====
# Download file from bucket S3
# -----
file_seagrass_site_data <- paste(conf_directory_temporary_data, param_file_seagrass_site_data, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_seagrass_site_data, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_seagrass_site_data)
    } else {
        file_path <- paste(param_user_email, param_file_seagrass_site_data, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_seagrass_site_data)
}

# Open datafile with seagrass site data
# -----
Seagrass_site <- readxl::read_excel(
    file_seagrass_site_data,
    sheet = "Data", 
    col_types = c("numeric", "numeric", "text")
)

# Convert seagrass_species to a factor variable
Seagrass_site$seagrass_species <- as.factor(Seagrass_site$seagrass_species)
summary(Seagrass_site)

In [ ]:
# Bottom_T_p95
# =====
# Download file from bucket S3
# -----
file_bottomT_p95 <- paste(conf_directory_temporary_data, param_file_bottomT_p95, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_bottomT_p95, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_bottomT_p95)
    } else {
        file_path <- paste(param_user_email, param_file_bottomT_p95, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_bottomT_p95)
}

# Open netcdf
# -----
bottomT_p95 <- open.nc(file_bottomT_p95)
# print.nc(bottomT_p95)

# Get data
# -----
bottomT_p95.value <- var.get.nc(bottomT_p95, "p95_bottomT_daily_C", unpack=TRUE)
bottomT_p95.lat   <- var.get.nc(bottomT_p95, "latitude")
bottomT_p95.lon   <- var.get.nc(bottomT_p95, "longitude")

# Check data
# -----
dim(bottomT_p95.value)
dim(bottomT_p95.lat)
dim(bottomT_p95.lon)

install.packages("raster")
library(raster)

no2df <- raster(
    bottomT_p95.value, 
    xmn=min(bottomT_p95.lon), xmx=max(bottomT_p95.lon), 
    ymn=min(bottomT_p95.lat), ymx=max(bottomT_p95.lat), 
    crs=CRS("+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs+ towgs84=0,0,0"))
plot(no2df)
# plt_df <- rotate(no2df)
# plot(plt_df)
# plt_df <- flip(plt_df, direction='y')
# plot(plt_df)

# no2df <- data.frame(
#     lat=as.vector(bottomT_p95.lat),
#     lon=as.vector(bottomT_p95.lon),
#     value=as.vector(bottomT_p95.value))
# ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
#   geom_point() +
#   labs(title = "p95_bottomT_daily_C",
#        x = "longitude",
#        y = "latitude")

# Close netcdf
# -----
close.nc(bottomT_p95)

print("Extract closest matching value based on location")
# -----
# This first function (extract_values) extracts values of matching sites, however this gives NAs for sites
# that appear over land given that the data product is at 0.083deg
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(bottomT_p95.lat - lat_value))
  lon_index <- which.min(abs(bottomT_p95.lon - lon_value))
  return(bottomT_p95.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site %>%
  rowwise() %>%
  mutate(bottomT_p95_C = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
# -----
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in bottomT_p95.value
  valid_indices <- which(!is.na(bottomT_p95.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- bottomT_p95.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- bottomT_p95.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value from bottomT_p95.value
  value_at_closest <- bottomT_p95.value[closest_valid_index[1], closest_valid_index[2]]
  
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(bottomT_p95_C_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = bottomT_p95_C_closest)) +
  geom_point() +
  labs(title = "bottomT_p95_C_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# Uo_mean
# =====
# Download file from bucket S3
# -----
file_uo_mean_1.5m_m_s <- paste(conf_directory_temporary_data, param_file_uo_mean_1.5m_m_s, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_uo_mean_1.5m_m_s, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_uo_mean_1.5m_m_s)
    } else {
        file_path <- paste(param_user_email, param_file_uo_mean_1.5m_m_s, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_uo_mean_1.5m_m_s)
}

# Open netcdf
# -----
uo_mean_1.5m_m_s <- open.nc(file_uo_mean_1.5m_m_s)
# print.nc(uo_mean_1.5m_m_s)

# Get data
uo_mean_1.5m.value <- var.get.nc(uo_mean_1.5m_m_s, 'uo_mean_1.5m_m_s', unpack=TRUE)
uo_mean_1.5m.lat <- var.get.nc(uo_mean_1.5m_m_s, 'latitude')
uo_mean_1.5m.lon <- var.get.nc(uo_mean_1.5m_m_s, 'longitude')

# Check data
no2df = NULL
no2df <- rbind(no2df, data.frame(
    lat=as.vector(uo_mean_1.5m.lat),
    lon=as.vector(uo_mean_1.5m.lon),
    value=as.vector(uo_mean_1.5m.value)))
ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
  geom_point() +
  labs(title = "uo_mean_1.5m_m_s",
       x = "longitude",
       y = "latitude")

# Close netcdf
close.nc(uo_mean_1.5m_m_s)

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(uo_mean_1.5m.lat - lat_value))
  lon_index <- which.min(abs(uo_mean_1.5m.lon - lon_value))
  return(uo_mean_1.5m.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(uo_mean_1.5m_m_s = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in uo_mean_1.5m.value
  valid_indices <- which(!is.na(uo_mean_1.5m.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- uo_mean_1.5m.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- uo_mean_1.5m.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- uo_mean_1.5m.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(uo_mean_1.5m_m_s_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = uo_mean_1.5m_m_s_closest)) +
  geom_point() +
  labs(title = "uo_mean_1.5m_m_s_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# Vo_p90 
# =====
# Download file from bucket S3
# -----
file_vo_p90_1.5m_m_s <- paste(conf_directory_temporary_data, param_file_vo_p90_1.5m_m_s, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_vo_p90_1.5m_m_s, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_vo_p90_1.5m_m_s)
    } else {
        file_path <- paste(param_user_email, param_file_vo_p90_1.5m_m_s, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_vo_p90_1.5m_m_s)
}

# Open netcdf
# -----
vo_p90_1.5m_m_s <- open.nc(file_vo_p90_1.5m_m_s)
# print.nc(vo_p90_1.5m_m_s)

# Get data
vo_p90_1.5m.value <- var.get.nc(vo_p90_1.5m_m_s, 'vo_p90_1.5m_m_s', unpack=TRUE)
vo_p90_1.5m.lat <- var.get.nc(vo_p90_1.5m_m_s, 'latitude')
vo_p90_1.5m.lon <- var.get.nc(vo_p90_1.5m_m_s, 'longitude')

# Check data
no2df = NULL
no2df <- rbind(no2df, data.frame(
    lat=as.vector(vo_p90_1.5m.lat),
    lon=as.vector(vo_p90_1.5m.lon),
    value=as.vector(vo_p90_1.5m.value)))
ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
  geom_point() +
  labs(title = "vo_p90_1.5m_m_s",
       x = "longitude",
       y = "latitude")

# Close netcdf
close.nc(vo_p90_1.5m_m_s)

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(vo_p90_1.5m.lat - lat_value))
  lon_index <- which.min(abs(vo_p90_1.5m.lon - lon_value))
  return(vo_p90_1.5m.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(vo_p90_1.5m_m_s = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in vo_p90_1.5m.value
  valid_indices <- which(!is.na(vo_p90_1.5m.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- vo_p90_1.5m.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- vo_p90_1.5m.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- vo_p90_1.5m.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(vo_p90_1.5m_m_s_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = vo_p90_1.5m_m_s_closest)) +
  geom_point() +
  labs(title = "vo_p90_1.5m_m_s_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# Phospate_mean
# =====
# Download file from bucket S3
# -----
file_po4_mean_1.5m_mmol_m3 <- paste(conf_directory_temporary_data, param_file_po4_mean_1.5m_mmol_m3, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_po4_mean_1.5m_mmol_m3, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_po4_mean_1.5m_mmol_m3)
    } else {
        file_path <- paste(param_user_email, param_file_po4_mean_1.5m_mmol_m3, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_po4_mean_1.5m_mmol_m3)
}

# Open netcdf
# -----
po4_mean_1.5m_mmol_m3 <- open.nc(file_po4_mean_1.5m_mmol_m3)
# print.nc(po4_mean_1.5m_mmol_m3)

# Get data
po4_mean.value <- var.get.nc(po4_mean_1.5m_mmol_m3, 'po4_mean_1.5m_mmol_m3', unpack=TRUE)
po4_mean.lat <- var.get.nc(po4_mean_1.5m_mmol_m3, 'latitude')
po4_mean.lon <- var.get.nc(po4_mean_1.5m_mmol_m3, 'longitude')

# Check data
no2df = NULL
no2df <- rbind(no2df, data.frame(
    lat=as.vector(po4_mean.lat),
    lon=as.vector(po4_mean.lon),
    value=as.vector(po4_mean.value)))
ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
  geom_point() +
  labs(title = "po4_mean_1.5m_mmol_m3",
       x = "longitude",
       y = "latitude")

# Close netcdf
close.nc(po4_mean_1.5m_mmol_m3) #close file

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(po4_mean.lat - lat_value))
  lon_index <- which.min(abs(po4_mean.lon - lon_value))
  return(po4_mean.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(po4_mean_1.5m_mmol_m3 = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in po4_mean.value
  valid_indices <- which(!is.na(po4_mean.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- po4_mean.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- po4_mean.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- po4_mean.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(po4_mean_1.5m_mmol_m3_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = po4_mean_1.5m_mmol_m3_closest)) +
  geom_point() +
  labs(title = "po4_mean_1.5m_mmol_m3_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# pH_mean
# =====
# Download file from bucket S3
# -----
file_pH_mean_1.5m <- paste(conf_directory_temporary_data, param_file_pH_mean_1.5m, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_pH_mean_1.5m, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_pH_mean_1.5m)
    } else {
        file_path <- paste(param_user_email, param_file_pH_mean_1.5m, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_pH_mean_1.5m)
}

# Open netcdf
# -----
pH_mean_1.5m <- open.nc(file_pH_mean_1.5m)
# print.nc(pH_mean_1.5m)

# Get data
pH_mean.value <- var.get.nc(pH_mean_1.5m, 'pH_mean_1.5m', unpack=TRUE)
pH_mean.lat <- var.get.nc(pH_mean_1.5m, 'latitude')
pH_mean.lon <- var.get.nc(pH_mean_1.5m, 'longitude')

# Check data
no2df = NULL
no2df <- rbind(no2df, data.frame(
    lat=as.vector(pH_mean.lat),
    lon=as.vector(pH_mean.lon),
    value=as.vector(pH_mean.value)))
ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
  geom_point() +
  labs(title = "pH_mean",
       x = "longitude",
       y = "latitude")

# Close netcdf
close.nc(pH_mean_1.5m)

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(pH_mean.lat - lat_value))
  lon_index <- which.min(abs(pH_mean.lon - lon_value))
  return(pH_mean.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(pH_mean_1.5m = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
# This second function (extract_closest_values) extracts values of the closest sites, excluding the matching
# sites that were NAs, and instead inputing the closest available value for these
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in pH_mean.value
  valid_indices <- which(!is.na(pH_mean.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- pH_mean.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- pH_mean.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- pH_mean.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(pH_mean_1.5m_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = pH_mean_1.5m_closest)) +
  geom_point() +
  labs(title = "pH_mean_1.5m_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# VHM0_p95
# =====
# Download file from bucket S3
# -----
file_wave_height_VHM0_p95_m <- paste(conf_directory_temporary_data, param_file_wave_height_VHM0_p95_m, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_wave_height_VHM0_p95_m, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_wave_height_VHM0_p95_m)
    } else {
        file_path <- paste(param_user_email, param_file_wave_height_VHM0_p95_m, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_wave_height_VHM0_p95_m)
}

# Open netcdf
# -----
wave_height_VHM0_p95_m <- open.nc(file_wave_height_VHM0_p95_m)
# print.nc(wave_height_VHM0_p95_m)

# Get data
VHM0_p95_m.value <- var.get.nc(wave_height_VHM0_p95_m, 'wave_height_VHM0_p95_m', unpack=TRUE)
VHM0_p95_m.lat <- var.get.nc(wave_height_VHM0_p95_m, 'latitude')
VHM0_p95_m.lon <- var.get.nc(wave_height_VHM0_p95_m, 'longitude')

# Check data
no2df = NULL
no2df <- rbind(no2df, data.frame(
    lat=as.vector(VHM0_p95_m.lat),
    lon=as.vector(VHM0_p95_m.lon),
    value=as.vector(VHM0_p95_m.value)))
ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
  geom_point() +
  labs(title = "wave_height_VHM0_p95_m",
       x = "longitude",
       y = "latitude")

# Close netcdf
close.nc(wave_height_VHM0_p95_m)

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(VHM0_p95_m.lat - lat_value))
  lon_index <- which.min(abs(VHM0_p95_m.lon - lon_value))
  return(VHM0_p95_m.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(wave_height_VHM0_p95_m = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in VHM0_p95_m.value
  valid_indices <- which(!is.na(VHM0_p95_m.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- VHM0_p95_m.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- VHM0_p95_m.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- VHM0_p95_m.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(wave_height_VHM0_p95_m_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = wave_height_VHM0_p95_m_closest)) +
  geom_point() +
  labs(title = "wave_height_VHM0_p95_m_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# fgCO2_p95
# =====
# Download file from bucket S3
# -----
file_Surf_fgco2_p95_molC_m2_yr <- paste(conf_directory_temporary_data, param_file_Surf_fgco2_p95_molC_m2_yr, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_Surf_fgco2_p95_molC_m2_yr, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_Surf_fgco2_p95_molC_m2_yr)
    } else {
        file_path <- paste(param_user_email, param_file_Surf_fgco2_p95_molC_m2_yr, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_Surf_fgco2_p95_molC_m2_yr)
}

# Open netcdf
# -----
Surf_fgco2_p95_molC_m2_yr <- open.nc(file_Surf_fgco2_p95_molC_m2_yr)
# print.nc(Surf_fgco2_p95_molC_m2_yr)

# Get data
Surf_fgco2_p95.value <- var.get.nc(Surf_fgco2_p95_molC_m2_yr, 'Surf_fgco2_p95_molC_m2_yr', unpack=TRUE)
Surf_fgco2_p95.lat <- var.get.nc(Surf_fgco2_p95_molC_m2_yr, 'latitude')
Surf_fgco2_p95.lon <- var.get.nc(Surf_fgco2_p95_molC_m2_yr, 'longitude')

# Check data
no2df = NULL
no2df <- rbind(no2df, data.frame(
    lat=as.vector(Surf_fgco2_p95.lat),
    lon=as.vector(Surf_fgco2_p95.lon),
    value=as.vector(Surf_fgco2_p95.value)))
ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
  geom_point() +
  labs(title = "Surf_fgco2_p95_molC_m2_yr",
       x = "longitude",
       y = "latitude")

# Close netcdf
close.nc(Surf_fgco2_p95_molC_m2_yr)

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(Surf_fgco2_p95.lat - lat_value))
  lon_index <- which.min(abs(Surf_fgco2_p95.lon - lon_value))
  return(Surf_fgco2_p95.value[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(Surf_fgco2_p95_molC_m2_yr = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in Surf_fgco2_p95.value
  valid_indices <- which(!is.na(Surf_fgco2_p95.value), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- Surf_fgco2_p95.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- Surf_fgco2_p95.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- Surf_fgco2_p95.value[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(Surf_fgco2_p95_molC_m2_yr_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = Surf_fgco2_p95_molC_m2_yr_closest)) +
  geom_point() +
  labs(title = "Surf_fgco2_p95_molC_m2_yr_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# KD490
# =====
# Download file from bucket S3
# -----
file_KD <- paste(conf_directory_temporary_data, param_file_KD, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_KD, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_KD)
    } else {
        file_path <- paste(param_user_email, param_file_KD, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_KD)
}

# Open netcdf
# -----
KD <- open.nc(file_KD)
# print.nc(KD)

# Get data
KD.Kd490 <- var.get.nc(KD, 'Kd490', unpack=TRUE)
KD.lat <- var.get.nc(KD, 'lat')
KD.lon <- var.get.nc(KD, 'lon')

# # Check data, can't plot
# no2df = NULL
# no2df <- rbind(no2df, data.frame(
#     lat=as.vector(KD.lat),
#     lon=as.vector(KD.lon),
#     value=as.vector(KD.Kd490)))
# ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
#   geom_point() +
#   labs(title = "KD",
#        x = "longitude",
#        y = "latitude")

# Close netcdf
close.nc(KD)

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(KD.lat - lat_value))
  lon_index <- which.min(abs(KD.lon - lon_value))
  return(KD.Kd490[lon_index, lat_index])
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(KD = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in KD.Kd490
  valid_indices <- which(!is.na(KD.Kd490), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- KD.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- KD.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value from KD.Kd490
  value_at_closest <- KD.Kd490[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(KD_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = KD_closest)) +
  geom_point() +
  labs(title = "KD_closest",
       x = "longitude",
       y = "latitude")

In [ ]:
# RRS443
# =====
# Download file from bucket S3
# -----
file_RRS443 <- paste(conf_directory_temporary_data, param_file_RRS443, sep="/")

if (param_use_dummy_data) {
        file_path <- paste(conf_minio_bucket_public_path, param_file_RRS443, sep="/")
        print(sprintf("Using dummy data for testing purposes. Set param_use_dummy_data to 0 to use your own data. Downloading data from %s / %s", conf_minio_bucket_public, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_public, object=file_path, file=file_RRS443)
    } else {
        file_path <- paste(param_user_email, param_file_RRS443, sep="/")
        print(sprintf("Downloading data from %s / %s", conf_minio_user_bucket, file_path))
        aws.s3::save_object(bucket=conf_minio_bucket_user, object=file_path, file=file_RRS443)
}

# Open netcdf
# -----
RRS443 <- open.nc(file_RRS443)
# print.nc(RRS443)

# Get data
RRS443.Rrs_443 <- var.get.nc(RRS443, 'Rrs_443', unpack=TRUE)
RRS443.lat <- var.get.nc(RRS443, 'lat')
RRS443.lon <- var.get.nc(RRS443, 'lon')

# # Check data, can't plot
# no2df = NULL
# no2df <- rbind(no2df, data.frame(
#     lat=as.vector(RRS443.lat),
#     lon=as.vector(RRS443.lon),
#     value=as.vector(RRS443.Rrs_443)))
# ggplot(data = no2df, aes(x = lon, y = lat, col = value)) +
#   geom_point() +
#   labs(title = "RRS443",
#        x = "longitude",
#        y = "latitude")

# Close netcdf
close.nc(RRS443)

# Start analysis
print("Extract closest matching value based on location")
extract_values <- function(lat_value, lon_value) {
  lat_index <- which.min(abs(RRS443.lat - lat_value))
  lon_index <- which.min(abs(RRS443.lon - lon_value))
  return(RRS443.Rrs_443[lon_index, lat_index])  # Adjust index order if needed
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(RRS443 = extract_values(latitude, longitude))

Seagrass_site_withEnv

print("Only select non-NA points")
extract_closest_values <- function(lat_value, lon_value) {
  # Get indices of valid (non-NA) points in RRS443.Rrs_443
  valid_indices <- which(!is.na(RRS443.Rrs_443), arr.ind = TRUE)
  
  # Extract the corresponding valid latitude and longitude values
  valid_Lat <- RRS443.lat[valid_indices[, 2]]  # lat corresponds to columns
  valid_Lon <- RRS443.lon[valid_indices[, 1]]  # lon corresponds to rows
  
  # Calculate distances to the target point for valid locations
  distances <- sqrt((valid_Lat - lat_value)^2 + (valid_Lon - lon_value)^2)
  
  # Find the index of the closest valid point
  closest_valid_index <- valid_indices[which.min(distances), ]
  
  # Retrieve the corresponding value
  value_at_closest <- RRS443.Rrs_443[closest_valid_index[1], closest_valid_index[2]]
  return(value_at_closest)
}

Seagrass_site_withEnv <- Seagrass_site_withEnv %>%
  rowwise() %>%
  mutate(RRS443_closest = extract_closest_values(latitude, longitude))

# Check data
# -----
Seagrass_site_withEnv

ggplot(data = Seagrass_site_withEnv, aes(x = longitude, y = latitude, col = RRS443_closest)) +
  geom_point() +
  labs(title = "RRS443_closest",
       x = "longitude",
       y = "latitude")

# Output

file_SG_modeling_dataframe = "data/SG_modeling_dataframe.csv"


In [ ]:
# Remove extra columns (should only retain the env. covariates with "_closest")
# =====
file_SG_modeling_dataframe <- paste(conf_directory_temporary_data, param_file_SG_modeling_dataframe, sep="/")

# Select data to save
# -----
Seagrass_site_withEnv <- Seagrass_site_withEnv %>% 
  select(-c("bottomT_p95_C", 
            "uo_mean_1.5m_m_s",
            "vo_p90_1.5m_m_s", 
            "po4_mean_1.5m_mmol_m3", 
            "pH_mean_1.5m", 
            "wave_height_VHM0_p95_m",
            "Surf_fgco2_p95_molC_m2_yr", 
            "KD", 
            "RRS443"))

# Some of the remote sensing variables can still have negative values even though they should not. 
# Set these to zero if this is the case.
Seagrass_site_withEnv$KD_closest[Seagrass_site_withEnv$KD_closest < 0] <- 0
Seagrass_site_withEnv$RRS443_closest[Seagrass_site_withEnv$RRS443_closest < 0] <- 0

# Add a variable for sediment mean depth
# For each row in the current dataframe, expand to 10 rows (with the same values), and add a new column called
# "sediment_mean_depth_cm" with the following entries "5, 15, 25, 35, 45, 55, 65, 75, 85, 95"
depths <- c(5, 15, 25, 35, 45, 55, 65, 75, 85, 95)
Seagrass_site_expanded <- Seagrass_site_withEnv[rep(1:nrow(Seagrass_site_withEnv), each = length(depths)), ]
Seagrass_site_expanded$sediment_mean_depth_cm <- rep(depths, times = nrow(Seagrass_site_withEnv))

# Check
# -----
# summary(Seagrass_site_expanded)
str(Seagrass_site_expanded) # all should be numeric, except for the seagrass_species factor variable

# Save dataframe to use for model prediction
# -----
write.csv(Seagrass_site_expanded, file_SG_modeling_dataframe, row.names = FALSE)